# CV Project: Object Detection + Action Recognition (PyTorch)

Run this notebook on **Google Colab with a GPU runtime** (Runtime -> Change runtime type -> GPU).

Two pipelines:
1. **Object detection** - COCO-pretrained Faster R-CNN, run frame-by-frame on any video you upload.
2. **Action recognition** - fine-tune a Kinetics-pretrained r3d_18 on a folder-per-class video dataset (UCF101/UCF11/HMDB51-style).

## 0. Setup

In [ ]:
!git clone https://github.com/YOUR_USERNAME/data-science-core.git
%cd data-science-core/projects/06-cv-detection-action-recognition
!pip install -q -r requirements.txt
import sys; sys.path.append('src')
import torch
print('CUDA available:', torch.cuda.is_available())

## 1. Object detection on a sample video

Upload your own video with the Colab file browser, or generate a synthetic test clip to confirm everything works first.

In [ ]:
from utils import generate_synthetic_video
generate_synthetic_video('data/sample.mp4', n_frames=60)

from object_detection import process_video, summarize_detections
df = process_video('data/sample.mp4', 'data/sample_annotated.mp4', every_n=3, score_thresh=0.5, pretrained=True)
summarize_detections(df)

Replace `data/sample.mp4` above with a real uploaded video for meaningful detections - the synthetic clip is just moving shapes and won't match any COCO class.

## 2. Action recognition: download a dataset

Recommended for Colab's free tier: **UCF11** (~1,600 short clips, 11 classes) is far more Colab-friendly than
full UCF101 (~13,000 clips). Get it via Kaggle:

```python
# 1. Upload your kaggle.json (Kaggle account -> Settings -> Create New API Token)
from google.colab import files
files.upload()  # select kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d kmader/ucf11-youtube-action -p data/ucf11 --unzip
```

Then confirm the folder structure is `data/ucf11/<class_name>/<video files>` (some Kaggle mirrors nest an extra folder level - flatten if needed).

In [ ]:
# Uncomment once the dataset is downloaded and confirmed at the expected path:
# from action_recognition import train
# model, classes = train('data/ucf11', epochs=5, batch_size=8, pretrained=True)

## 3. (Optional) Smoke-test action recognition without downloading a dataset

Generates a tiny synthetic 2-class video set so you can confirm the training loop runs before committing to a full dataset download.

In [ ]:
import os
from utils import generate_synthetic_video
for cls in ['moving_right', 'bouncing']:
    os.makedirs(f'data/mini_dataset/{cls}', exist_ok=True)
    for i in range(3):
        generate_synthetic_video(f'data/mini_dataset/{cls}/vid_{i}.mp4', n_frames=20)

from action_recognition import train
model, classes = train('data/mini_dataset', epochs=2, batch_size=2, pretrained=True)